# 🦀 Day 2: Unsafe Functions and Traits
---

Today we dive deeper into **unsafe functions** and **unsafe traits**.

- **`unsafe fn`**: Functions with preconditions the compiler can't verify
- **`# Safety` doc comments**: Documenting what callers must guarantee
- **`unsafe trait`** and **`unsafe impl`**: e.g., `Send`, `Sync`
- **The unsafe sandwich**: Safe wrapper → unsafe core → safe wrapper
- **`std::ptr`**: `read`, `write`, `copy`, `null`, `null_mut`
- **`NonNull<T>`**: Non-null raw pointer
- **Common UB**: Undefined behavior to avoid

In [ ]:
// unsafe fn: callers must uphold invariants
/// # Safety
/// Caller must ensure `ptr` is valid, non-null, and properly aligned.
/// The pointer must point to an initialized value of type T.
unsafe fn read_unchecked<T>(ptr: *const T) -> T {
    std::ptr::read(ptr)
}

let x = 42i32;
let ptr = &x as *const i32;
let value = unsafe { read_unchecked(ptr) };
println!("Read value: {}", value);

In [ ]:
// std::ptr module: read, write, copy, null, null_mut
use std::ptr;

let mut a = 10i32;
let mut b = 20i32;
let pa = &mut a as *mut i32;
let pb = &mut b as *mut i32;

unsafe {
    let val = ptr::read(pa);
    ptr::write(pb, val);
}
println!("a={}, b={}", a, b);

// null and null_mut for sentinel values
let null_ptr: *const i32 = ptr::null();
println!("Is null: {}", null_ptr.is_null());

In [ ]:
// NonNull<T>: guaranteed non-null raw pointer
use std::ptr::NonNull;

let x = 42;
let ptr = NonNull::new(&x as *const i32 as *mut i32).unwrap();
unsafe {
    println!("NonNull value: {}", *ptr.as_ptr());
}

// NonNull is used in allocators and collections (e.g., Box, Vec)
// because null can mean "no allocation" — NonNull encodes "we have one"

In [ ]:
// unsafe trait: implementing it requires unsafe impl
// Send and Sync are unsafe traits — wrong impl = data races
struct MySendType(i32);

unsafe impl Send for MySendType {}
unsafe impl Sync for MySendType {}

// Safe wrapper over unsafe: the "unsafe sandwich"
fn safe_read<T: Copy>(ptr: *const T) -> Option<T> {
    if ptr.is_null() {
        None
    } else {
        unsafe { Some(ptr::read(ptr)) }
    }
}

let x = 99;
println!("safe_read: {:?}", safe_read(&x));
println!("safe_read null: {:?}", safe_read(ptr::null()));

## 📝 Exercise: Safe abstraction over unsafe memory operation

Write a safe function `copy_range<T: Copy>(src: &[T], dst: &mut [T], src_start: usize, dst_start: usize, len: usize)` that copies `len` elements from `src[src_start..]` to `dst[dst_start..]` using `ptr::copy` internally. Return `Result<(), &'static str>` on bounds errors.

In [ ]:
// YOUR CODE HERE

## 🎯 Key Takeaways

1. **`unsafe fn`** requires callers to uphold documented invariants; always add `# Safety`
2. **`std::ptr`**: `read`, `write`, `copy`, `null`, `null_mut` for low-level ops
3. **`NonNull<T>`** guarantees non-null; useful for allocators and collections
4. **`unsafe trait`** + **`unsafe impl`**: e.g., `Send`, `Sync` — wrong impl = UB
5. **Safe abstractions** wrap unsafe in safe APIs — the "unsafe sandwich"

---
**Tomorrow:** FFI basics — calling C from Rust